# 🔍 Universal Lead Finder — L&D Designs
Finds **any type of local business** with no website within 50 miles of Wigan.

**How to use:**
1. In Cell 1 — pick your business type (or add a custom one)
2. Run Cell 2 — searches OpenStreetMap + Yell.com
3. Run Cell 3 — saves your spreadsheet

In [ ]:
# ══════════════════════════════════════════════════════════════
#  Cell 1 — PICK YOUR BUSINESS TYPE
#  Change BUSINESS_TYPE to any key from the list below,
#  or scroll down to add your own custom one.
# ══════════════════════════════════════════════════════════════

BUSINESS_TYPE = 'restaurants'   # <-- CHANGE THIS

# ── Available presets ─────────────────────────────────────────
PRESETS = {

    'barbers': {
        'label': 'Barbers & Hairdressers',
        'yell_keywords': ['barbers', 'hairdressers', 'hair salon', 'barbershop'],
        'osm_tags': [
            ('shop', 'hairdresser'), ('shop', 'barber'), ('amenity', 'hairdresser')
        ],
    },

    'restaurants': {
        'label': 'Restaurants, Cafes & Takeaways',
        'yell_keywords': ['restaurant', 'takeaway', 'cafe', 'fish and chips', 'pizza'],
        'osm_tags': [
            ('amenity', 'restaurant'), ('amenity', 'fast_food'),
            ('amenity', 'cafe'), ('amenity', 'pub'),
        ],
    },

    'trades': {
        'label': 'Tradespeople (Plumbers, Electricians, Builders)',
        'yell_keywords': ['plumber', 'electrician', 'builder', 'plasterer', 'decorator'],
        'osm_tags': [
            ('craft', 'plumber'), ('craft', 'electrician'),
            ('craft', 'painter'), ('craft', 'builder'),
        ],
    },

    'beauty': {
        'label': 'Beauty & Nail Salons',
        'yell_keywords': ['beauty salon', 'nail salon', 'nail bar', 'lashes', 'aesthetics'],
        'osm_tags': [
            ('shop', 'beauty'), ('shop', 'cosmetics'), ('leisure', 'tanning'),
        ],
    },

    'garages': {
        'label': 'Car Garages & MOT Centres',
        'yell_keywords': ['car garage', 'MOT centre', 'car repair', 'mechanic', 'tyre fitting'],
        'osm_tags': [
            ('shop', 'car_repair'), ('amenity', 'car_wash'),
            ('shop', 'tyres'), ('amenity', 'fuel'),
        ],
    },

    'cleaning': {
        'label': 'Cleaning Companies',
        'yell_keywords': ['cleaning company', 'domestic cleaner', 'office cleaning', 'carpet cleaning'],
        'osm_tags': [],  # Not well tagged on OSM — Yell only
    },

    'landscaping': {
        'label': 'Landscapers & Gardeners',
        'yell_keywords': ['landscaper', 'gardener', 'garden maintenance', 'tree surgeon', 'lawn care'],
        'osm_tags': [
            ('craft', 'gardener'), ('craft', 'landscaper'),
        ],
    },

    'tattoo': {
        'label': 'Tattoo & Piercing Studios',
        'yell_keywords': ['tattoo studio', 'tattoo parlour', 'piercing studio'],
        'osm_tags': [
            ('shop', 'tattoo'),
        ],
    },

    'gyms': {
        'label': 'Gyms & Personal Trainers',
        'yell_keywords': ['gym', 'personal trainer', 'fitness studio', 'boxing gym', 'yoga studio'],
        'osm_tags': [
            ('leisure', 'fitness_centre'), ('leisure', 'sports_centre'),
            ('amenity', 'gym'),
        ],
    },

    'pet': {
        'label': 'Dog Groomers & Pet Services',
        'yell_keywords': ['dog groomer', 'dog grooming', 'pet grooming', 'dog walker'],
        'osm_tags': [
            ('shop', 'pet_grooming'), ('amenity', 'veterinary'),
        ],
    },

    'accountants': {
        'label': 'Accountants & Bookkeepers',
        'yell_keywords': ['accountant', 'bookkeeper', 'tax advisor', 'chartered accountant'],
        'osm_tags': [
            ('office', 'accountant'),
        ],
    },

    'dentists': {
        'label': 'Dentists & Dental Practices',
        'yell_keywords': ['dentist', 'dental practice', 'dental surgery'],
        'osm_tags': [
            ('amenity', 'dentist'),
        ],
    },

    # ── ADD YOUR OWN CUSTOM TYPE HERE ──────────────────────────
    # 'custom': {
    #     'label': 'My Custom Business',
    #     'yell_keywords': ['keyword1', 'keyword2'],
    #     'osm_tags': [('shop', 'tag_value')],
    # },
}

# ─────────────────────────────────────────────────────────────
if BUSINESS_TYPE not in PRESETS:
    print('ERROR: Unknown type. Available types:')
    for k, v in PRESETS.items():
        print(' ', k, '—', v['label'])
else:
    cfg = PRESETS[BUSINESS_TYPE]
    print('✅ Selected:', cfg['label'])
    print('   Yell keywords:', cfg['yell_keywords'])
    print('   OSM tags:', cfg['osm_tags'])

In [ ]:
# ── Cell 2: Search ────────────────────────────────────────────────────────
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
                       'requests', 'beautifulsoup4', 'openpyxl', 'lxml'])

import re, math, time, unicodedata
from datetime import datetime
from urllib.parse import urljoin
import requests
from bs4 import BeautifulSoup

WIGAN_LAT  = 53.5450
WIGAN_LNG  = -2.6325
RADIUS_MI  = 50
RADIUS_M   = int(RADIUS_MI * 1609.344)
OUTDATED_YEARS = 3
SOCIAL_DOMAINS = ('facebook.com','instagram.com','twitter.com','tiktok.com','linkedin.com','linktree.com')
BROWSER_HEADERS = {'User-Agent':'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36'}
YELL_HEADERS = {**BROWSER_HEADERS,'Accept':'text/html,application/xhtml+xml;q=0.9,*/*;q=0.8','Accept-Language':'en-GB,en;q=0.9','Referer':'https://www.yell.com/'}

def haversine_mi(lat1,lng1,lat2,lng2):
    R=3958.8; p1,p2=math.radians(lat1),math.radians(lat2)
    dp=math.radians(lat2-lat1); dl=math.radians(lng2-lng1)
    a=math.sin(dp/2)**2+math.cos(p1)*math.cos(p2)*math.sin(dl/2)**2
    return 2*R*math.asin(math.sqrt(a))

def _osm_address(tags):
    parts=[tags.get('addr:housenumber',''),tags.get('addr:street',''),
           tags.get('addr:city','') or tags.get('addr:town',''),tags.get('addr:postcode','')]
    return ', '.join(p for p in parts if p)

def fetch_from_overpass(osm_tags):
    if not osm_tags:
        print('  No OSM tags for this type — skipping Overpass.')
        return []
    tag_lines = ''
    for k, v in osm_tags:
        tag_lines += 'node["'+k+'"="'+v+'"](around:'+str(RADIUS_M)+','+str(WIGAN_LAT)+','+str(WIGAN_LNG)+');\n'
        tag_lines += 'way["'+k+'"="'+v+'"](around:'+str(RADIUS_M)+','+str(WIGAN_LAT)+','+str(WIGAN_LNG)+');\n'
    query = '[out:json][timeout:90];\n(\n' + tag_lines + ');\nout center tags;'
    print('  Querying OpenStreetMap...', end=' ', flush=True)
    try:
        resp = requests.post('https://overpass-api.de/api/interpreter',
                             data={'data':query}, headers={'User-Agent':'WiganLeadFinder/1.0'}, timeout=120)
        resp.raise_for_status()
    except Exception as e:
        print('Error:', e); return []
    elements = resp.json().get('elements', [])
    print(len(elements), 'elements.')
    results = []
    for el in elements:
        tags = el.get('tags', {}); name = tags.get('name','').strip()
        if not name: continue
        if el['type'] == 'node': blat,blng = el.get('lat'),el.get('lon')
        else:
            c = el.get('center',{}); blat,blng = c.get('lat'),c.get('lon')
        if blat is None: continue
        dist = haversine_mi(WIGAN_LAT,WIGAN_LNG,blat,blng)
        if dist > RADIUS_MI: continue
        results.append({'name':name,'phone':tags.get('phone') or tags.get('contact:phone',''),
                        'website':tags.get('website') or tags.get('contact:website',''),
                        'address':_osm_address(tags),'rating':'','reviews':0,
                        'distance_mi':round(dist,1),'source':'osm'})
    print('  OSM:', len(results), 'businesses within', RADIUS_MI, 'miles.')
    return results

def fetch_from_yell(yell_keywords):
    all_results = []
    for keyword in yell_keywords:
        print('  Yell:', repr(keyword), '', end='', flush=True)
        for page_num in range(1, 21):
            try:
                resp = requests.get('https://www.yell.com/ucs/UcsSearchAction.do',
                    params={'keywords':keyword,'location':'Wigan, Greater Manchester','radius':'50','pageNum':str(page_num)},
                    headers=YELL_HEADERS, timeout=20)
                if resp.status_code != 200: break
                html = resp.text
            except: break
            soup = BeautifulSoup(html, 'lxml')
            arts = soup.find_all('article', class_=lambda c: c and 'businessCapsule--listing' in c)
            if not arts: break
            for art in arts:
                def _t(cls): tag=art.find(class_=lambda c:c and cls in c); return tag.get_text(strip=True) if tag else ''
                name = _t('businessCapsule--name')
                if not name: continue
                site_tag = art.find('a', class_=lambda c: c and 'businessCapsule--website' in c)
                website = (site_tag.get('data-url') or site_tag.get('href','')) if site_tag else ''
                dist_txt = _t('businessCapsule--distance')
                dist_mi = 0.0
                m = re.search(r'([\d.]+)\s*miles?', dist_txt, re.I)
                if m: dist_mi = float(m.group(1))
                if dist_mi <= RADIUS_MI:
                    all_results.append({'name':name,'phone':_t('businessCapsule--telephone'),
                                        'website':website,'address':_t('businessCapsule--address'),
                                        'rating':'','reviews':0,'distance_mi':dist_mi,'source':'yell'})
            print('.', end='', flush=True)
            if len(arts) < 10: break
            time.sleep(1.5)
        print()
    print('  Yell total:', len(all_results))
    return all_results

_PC_RE = re.compile(r'\b([A-Z]{1,2}\d{1,2}[A-Z]?\s?\d[A-Z]{2})\b', re.I)
def _norm(s): s=unicodedata.normalize('NFKD',s).encode('ascii','ignore').decode(); return re.sub(r'[^a-z0-9]','',s.lower())
def _postcode(addr): m=_PC_RE.search(addr); return _norm(m.group(1)) if m else ''
def deduplicate(a, b):
    seen={}; merged=[]
    for biz in a+b:
        words=biz['name'].split(); first=_norm(words[0]) if words else ''
        pc=_postcode(biz['address']); key=(first,pc) if pc else (_norm(biz['name']),)
        if key not in seen: seen[key]=True; merged.append(biz)
    return merged

def check_website(url):
    if not url: return 'none','No website listed'
    if any(d in url.lower() for d in SOCIAL_DOMAINS):
        d=next(x for x in SOCIAL_DOMAINS if x in url.lower()); return 'social_only','Only a '+d+' page'
    try:
        resp=requests.get(url,headers=BROWSER_HEADERS,timeout=12,allow_redirects=True)
    except requests.exceptions.SSLError: return 'outdated','Broken SSL'
    except requests.exceptions.ConnectionError: return 'error','Cannot connect'
    except requests.exceptions.Timeout: return 'error','Timed out'
    except Exception as e: return 'error',str(e)[:60]
    if resp.status_code>=400: return 'error','HTTP '+str(resp.status_code)
    cur=datetime.now().year; text=resp.text
    lm=resp.headers.get('Last-Modified','')
    if lm:
        try:
            from email.utils import parsedate; p=parsedate(lm)
            if p and cur-p[0]>=OUTDATED_YEARS: return 'outdated','Last-Modified: '+str(p[0])
        except: pass
    yrs=re.findall(r'copyright[^\d]{0,10}(\d{4})',text,re.I)+re.findall(r'[©]\s*(\d{4})',text)
    valid=[int(y) for y in yrs if 2000<=int(y)<=cur+1]
    if valid and cur-max(valid)>=OUTDATED_YEARS: return 'outdated','Copyright: '+str(max(valid))
    soup=BeautifulSoup(text,'lxml')
    gen=soup.find('meta',{'name':'generator'})
    if gen:
        c=gen.get('content','').lower(); wp=re.search(r'wordpress\s+([\d.]+)',c)
        if wp:
            try:
                if float(wp.group(1))<5.0: return 'outdated','Old WordPress '+wp.group(1)
            except: pass
    all_tags=soup.find_all()
    if all_tags and len(soup.find_all('table'))/len(all_tags)>0.12: return 'outdated','Table-based layout'
    return 'active','Website looks current'

EMAIL_RE=re.compile(r'\b[A-Za-z0-9._%+\-]+@[A-Za-z0-9.\-]+\.[A-Za-z]{2,}\b')
SKIP_EMAIL={'noreply','no-reply','example','test','wordpress','sentry','privacy','abuse','postmaster'}
WA_HREF_RE=re.compile(r'wa\.me/(\+?[\d]+)|whatsapp\.com/send\?phone=([\d+]+)',re.I)

def _get_html(url):
    try:
        r=requests.get(url,headers=BROWSER_HEADERS,timeout=10,allow_redirects=True)
        if r.status_code==200: return r.text
    except: pass
    return ''

def scrape_contacts(url):
    if not url or any(d in url.lower() for d in SOCIAL_DOMAINS): return '',''
    email=whatsapp=''
    for page_url in [url]+[urljoin(url,s) for s in ('/contact','/contact-us','/about')]:
        if email and whatsapp: break
        html=_get_html(page_url)
        if not html: continue
        soup=BeautifulSoup(html,'lxml')
        if not email:
            for tag in soup.find_all('a',href=re.compile(r'^mailto:',re.I)):
                m=re.search(r'mailto:([^\?&\s]+)',tag['href'])
                if m:
                    c=m.group(1).strip().lower()
                    if not any(s in c for s in SKIP_EMAIL): email=c; break
        if not email:
            for c in EMAIL_RE.findall(html):
                if not any(s in c.lower() for s in SKIP_EMAIL): email=c.lower(); break
        if not whatsapp:
            for tag in soup.find_all('a',href=True):
                m=WA_HREF_RE.search(tag['href'])
                if m: whatsapp=(m.group(1) or m.group(2)).strip(); break
    return email,whatsapp

# ── RUN ───────────────────────────────────────────────────────
label = cfg['label']
print('='*55)
print('Searching for:', label)
print('='*55)

print('\n[1/3] Fetching from OpenStreetMap + Yell.com...')
osm  = fetch_from_overpass(cfg['osm_tags'])
yell = fetch_from_yell(cfg['yell_keywords'])
raw  = deduplicate(osm, yell)
print('\n', len(raw), 'unique businesses ('+str(len(osm))+' OSM + '+str(len(yell))+' Yell)')

print('\n[2/3] Checking websites...')
leads = []
for idx,biz in enumerate(raw,1):
    status,notes = check_website(biz['website'])
    if idx % 25 == 0: print('  '+str(idx)+'/'+str(len(raw))+' checked...')
    if status == 'active': continue
    email,whatsapp = '',''
    if biz['website'] and status != 'none':
        email,whatsapp = scrape_contacts(biz['website'])
    leads.append({**biz,'status':status,'notes':notes,'email':email,'whatsapp':whatsapp})

print('\n', len(leads), 'leads found ('+str(len(raw)-len(leads))+' skipped — had active website)')
print('Done with checking!')

In [ ]:
# ── Cell 3: Save spreadsheet ──────────────────────────────────────────────
import openpyxl
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils import get_column_letter
from google.colab import files

COLS=[('Business Name',30),('Phone',18),('Email',32),('WhatsApp',18),
      ('Status',16),('Website/Social',42),('Address',45),('Notes',36),('Distance (mi)',14)]
FILL_H=PatternFill('solid',fgColor='1A2035')
FILLS={'none':PatternFill('solid',fgColor='FFD6D6'),'social_only':PatternFill('solid',fgColor='D6EAFF'),
       'outdated':PatternFill('solid',fgColor='FFF2CC'),'error':PatternFill('solid',fgColor='E8E8E8')}
THIN=Border(**{s:Side(style='thin',color='CCCCCC') for s in ('left','right','top','bottom')})
ORDER={'none':0,'social_only':1,'outdated':2,'error':3}

wb=openpyxl.Workbook(); ws=wb.active
business_slug = re.sub(r'[^a-z0-9]','_', label.lower())[:20]
ws.title='Leads'

for col,(hdr,w) in enumerate(COLS,1):
    c=ws.cell(1,col,hdr); c.fill=FILL_H; c.font=Font(color='FFFFFF',bold=True,size=10)
    c.alignment=Alignment(horizontal='center',vertical='center'); c.border=THIN
    ws.column_dimensions[get_column_letter(col)].width=w
ws.row_dimensions[1].height=28

sorted_leads=sorted(leads,key=lambda b:(ORDER.get(b['status'],9),b['distance_mi']))
for ri,biz in enumerate(sorted_leads,2):
    fill=FILLS.get(biz['status'],PatternFill(fill_type=None))
    for col,val in enumerate([biz['name'],biz['phone'],biz['email'],biz['whatsapp'],
                               biz['status'].upper().replace('_',' '),biz['website'],
                               biz['address'],biz['notes'],biz['distance_mi']],1):
        c=ws.cell(ri,col,val); c.fill=fill
        c.alignment=Alignment(vertical='center'); c.border=THIN
    ws.row_dimensions[ri].height=17
ws.freeze_panes='A2'; ws.auto_filter.ref=ws.dimensions

ts=datetime.now().strftime('%Y%m%d_%H%M%S')
fname='leads_'+business_slug+'_'+ts+'.xlsx'
wb.save(fname)
files.download(fname)

print('✅ Saved', len(sorted_leads), 'leads to', fname)
print()
print('Breakdown:')
for s in ('none','social_only','outdated','error'):
    n = sum(1 for b in sorted_leads if b['status']==s)
    if n: print(' ', s, ':', n)
print('With phone:', sum(1 for b in sorted_leads if b['phone']))
print('With email:', sum(1 for b in sorted_leads if b['email']))